In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib
import os

print("🧬 Loading Dataset into memory...")
# Automatically grab the extracted CSV file
csv_file = [f for f in os.listdir('.') if f.endswith('.csv')][0]
df = pd.read_csv(csv_file)
print(f"✅ Dataset loaded! Total patient records: {len(df)}")

# --- 1. SMART COLUMN MATCHER ---
# This ensures we grab the right columns even if the Kaggle author used spaces or weird caps
def find_column(dataframe, keyword):
    for col in dataframe.columns:
        if keyword.lower() in col.lower(): return col
    return None

target_col = find_column(df, 'treatment')
stage_col = find_column(df, 'stage')

# Looking for specific symptoms
symptom_keywords = ['cough', 'breath', 'chest', 'weight', 'fatigue']
symptom_cols = [find_column(df, kw) for kw in symptom_keywords if find_column(df, kw)]

# Keep only the columns we need and drop empty rows
features = [stage_col] + symptom_cols
df_clean = df[features + [target_col]].dropna().copy()

# --- 2. DATA STANDARDIZATION ---
# We must force the Kaggle stages to match our U-Net output perfectly
def standardize_stage(val):
    val = str(val).lower()
    if '4' in val or 'iv' in val: return "Stage IV"
    if '3' in val or 'iii' in val: return "Stage III"
    if '2' in val or 'ii' in val: return "Stage II"
    if '1' in val or 'i' in val: return "Stage I"
    return "Normal"

df_clean[stage_col] = df_clean[stage_col].apply(standardize_stage)

# Standardize symptoms to strictly "yes" or "no"
for col in symptom_cols:
    df_clean[col] = df_clean[col].astype(str).str.lower().apply(
        lambda x: "yes" if x in ['yes', '1', '1.0', 'true', 'y', 't'] else "no"
    )

# --- 3. TEXT-TO-NUMBER ENCODING ---
label_encoders = {}
for col in features:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le

target_le = LabelEncoder()
df_clean[target_col] = target_le.fit_transform(df_clean[target_col].astype(str))
label_encoders['TARGET'] = target_le

# --- 4. TRAIN XGBOOST ---
print("\n🧠 Training XGBoost Classifier...")
X = df_clean[features]
y = df_clean[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model = xgb.XGBClassifier(
    objective='multi:softprob', 
    eval_metric='mlogloss', 
    max_depth=6, 
    learning_rate=0.1, 
    n_estimators=100,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Evaluate the model
y_pred = xgb_model.predict(X_test)
print(f"✅ XGBoost Training Complete! Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

# --- 5. EXPORT FILES ---
os.makedirs('models', exist_ok=True)
xgb_model.save_model('models/xgboost_treatment_model.json')

xgboost_metadata = {
    'encoders': label_encoders,
    'features': features,
    'stage_col': stage_col,
    'symptom_cols': symptom_cols
}
joblib.dump(xgboost_metadata, 'models/xgboost_metadata.joblib')

print("📦 Model (.json) and Metadata (.joblib) successfully saved to the 'models' folder!")